# Locate Test Blocks by Program (v1)

**Author:** Aaron Roodman  
**Date Created:** 2026-01-27  
**Last Modified:** 2026-03-14  
**Status:** In Progress  
**Keywords:** Program, Test Blocks, Observing Blocks, ConsDB  

## Description

Queries the ConsDB Visit1 table for all observations in a date range, maps BLOCK test case IDs
to human-readable names via the Zephyr Scale API, and tabulates band counts by day_obs for
selected test blocks.

Key functionality:
1. Query all Visit1 data from ConsDB for the specified date range
2. Look up Zephyr Scale test case names for each BLOCK science_program
3. Print band-count tables by day_obs for selected test blocks

**Output:** `test_block_names.txt` — mapping of BLOCK IDs to test case names  
**Based on:** Code from Lynne Jones for Zephyr Scale lookups

## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-01-27 | Aaron Roodman | Initial version |
| 2026-03-14 | Aaron Roodman | Restructured to template; extended date range to 2026-03-14 |

## Table of Contents

1. [Parameters](#params)
2. [Setup & Imports](#setup)
3. [Helper Functions](#functions)
4. [Data Access](#data)
5. [Analysis](#analysis)
6. [Results & Plots](#results)

<a id='params'></a>
## Parameters

In [ ]:
# ============================================================
# Parameters — All configurable values collected here
# ============================================================

day_obs_min = 20250415
day_obs_max = 20260315

instrument = 'lsstcam'
bands = {'u': '#0c71ff', 'g': '#49be61', 'r': '#c61c00', 'i': '#ffc200', 'z': '#f341a2', 'y': '#5d0000'}

output_dir = 'output'

<a id='setup'></a>
## Setup & Imports

In [ ]:
import os
import sys
import re
from pathlib import Path

os.environ["no_proxy"] += ",.consdb"

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import requests

from astropy.time import Time, TimeDelta
from lsst.obs.lsst import LsstCam
from lsst.summit.utils import (
    ConsDbClient,
    getAirmassSeeingCorrection,
    getBandpassSeeingCorrection,
)
from lsst.summit.utils.efdUtils import (
    getEfdData,
    getMostRecentRowWithDataBefore,
    makeEfdClient,
)

# Add repo root to path for common imports
sys.path.insert(0, str(Path.cwd().parent))
from common.utils import setup_plotting

setup_plotting()

<a id='functions'></a>
## Helper Functions

In [ ]:
def dayObsToMJD(day_obs):
    year = np.floor(day_obs / 10000).astype(int)
    month = np.floor((day_obs % 10000) / 100).astype(int)
    day = day_obs % 100
    time = Time({'year': year, 'month': month, 'day': day}, format='ymdhms')
    return time.mjd

def MJDToDayObs(mjd):
    time = Time(mjd, format='mjd')
    return np.array([_.iso.split()[0].replace('-', '') for _ in time])    

In [ ]:
def print_band_counts_by_day(df, block_names, img_type_value):
    """
    Print a table showing band counts by day_obs for filtered data.
    
    Parameters
    ----------
    df : pandas.DataFrame
        The input DataFrame
    block_names : str or list of str
        Substring(s) to match in science_program column
    img_type_value : str
        Value to match in img_type column
    """
    # Convert single string to list for consistent handling
    if isinstance(block_names, str):
        block_names = [block_names]
    
    # Filter for specified blocks and img_type
    # Create mask for any of the block names
    block_mask = df['science_program'].str.contains(block_names[0], na=False)
    for block_name in block_names[1:]:
        block_mask |= df['science_program'].str.contains(block_name, na=False)
    
    filtered = df[block_mask & (df['img_type'] == img_type_value)].copy()
    
    print(f"Total rows matching {block_names} and img_type='{img_type_value}': {len(filtered)}")
    
    if len(filtered) == 0:
        print("No matching rows found.")
        return
    
    # Extract first character of band
    filtered['band_first'] = filtered['band'].str[0]
    
    # Create crosstab with day_obs as rows and first character of band as columns
    band_table = pd.crosstab(filtered['day_obs'], filtered['band_first'])
    
    # Reorder columns to u, g, r, i, z, y if they exist
    desired_order = ['u', 'g', 'r', 'i', 'z', 'y']
    band_table = band_table.reindex(columns=desired_order, fill_value=0)
    
    # Add totals row
    totals = band_table.sum(axis=0)
    band_table.loc['TOTAL'] = totals
    
    print("\nBand counts by day_obs:")
    print(band_table)

# Example usage:
# print_band_counts_by_day(df, 'BLOCK', 'SKYEXP')
# print_band_counts_by_day(df, ['BLOCK-123', 'BLOCK-456'], 'SKYEXP')

<a id='data'></a>
## Data Access

In [ ]:
# Translate test block numbers above into more meaningful programs
jira_base_url = "https://rubinobs.atlassian.net/projects/BLOCK?selectedItem=com.atlassian.plugins.atlassian-connect-plugin:com.kanoah.test-manager__main-project-page#!/v2/testCase/"
with open("/home/r/roodman/.lsst/zephyr_token", "r") as f:
    zephyr_token = f.read().rstrip("\n")
zephyr_url = "https://api.zephyrscale.smartbear.com/v2/testcases/"
headers = {"Accept": "application/json",
           "Authorization": f"Bearer {zephyr_token}",
           "Content-Type": "application/json"} 

# note that this token is only good for 1 year.  go to https://rubinobs.atlassian.net/plugins/servlet/ac/com.kanoah.test-manager/api-access-tokens?project.key=BLOCK&project.id=10064
# to renew

In [ ]:
client = ConsDbClient("http://consdb-pq.consdb:8080/consdb")

In [ ]:
# get all Visit1 info in this day_obs range, joined with quicklook for physical_rotator_angle
visits_query = f'''
    SELECT 
        v1.*,
        ql.physical_rotator_angle
    FROM 
        cdb_{instrument}.visit1 v1
    LEFT JOIN
        cdb_{instrument}.visit1_quicklook ql
    ON v1.visit_id = ql.visit_id
    WHERE 
        v1.day_obs >= {day_obs_min} AND v1.day_obs <= {day_obs_max}
'''

visits = client.query(visits_query).to_pandas()

In [ ]:
# some notes:
# scheduler_note is Observation Annotation in RubinTV

print(sorted(visits.columns))

In [ ]:
# code from Lynne Jones to get the Zephyr scale title for each science_program
if len(visits) > 0:
    test_names = {}
    jira_urls = {}
    for science_program in visits.science_program.unique():
        if science_program is not None and science_program.startswith("BLOCK-T"):
            # We have science_programs of the form BLOCK-T539_hexapods or _v3 .. anything post '_' is unusable
            response = requests.get(url=zephyr_url + science_program.split("_")[0], headers=headers)
            test_name = response.json()['name']
            jira_url = jira_base_url + science_program
            
            test_names[science_program] = test_name
            jira_urls[science_program] = jira_url

<a id='analysis'></a>
## Analysis

In [ ]:
program_list = visits['science_program'].value_counts().sort_index()

In [ ]:
# To display all entries temporarily for one Series
with pd.option_context('display.max_rows', None):
    print(program_list)

### Filter LUT BLOCK-T559

In [ ]:
print_band_counts_by_day(visits, 'T559', 'acq')

In [ ]:
print_band_counts_by_day(visits, 'T559', 'cwfs')

### BLOCK-T660 FAM Donuts during FBS

In [ ]:
print_band_counts_by_day(visits, 'T630', 'cwfs')

###  FAM Used in Analysis from Bryce

In [ ]:
programs_touse = ['T278','T381','T492','T539','T614']
with pd.option_context('display.max_rows', None):
    print_band_counts_by_day(visits, programs_touse, 'cwfs')

###  Stability in-focus

In [ ]:
programs_touse = ['T614']
with pd.option_context('display.max_rows', None):
    print_band_counts_by_day(visits, programs_touse, 'engtest')

<a id='results'></a>
## Results & Plots

In [ ]:
# Group by prefix and get one example value
prefix_dict = {}
for key, value in test_names.items():
    # Remove _vN or _N pattern from the end
    prefix = re.sub(r'_v?\d+$', '', key)
    if prefix not in prefix_dict:
        prefix_dict[prefix] = value

# Write sorted prefixes with values to output file and print
os.makedirs(output_dir, exist_ok=True)
output_file = os.path.join(output_dir, 'test_block_names.txt')
with open(output_file, 'w') as f:
    for prefix in sorted(prefix_dict.keys()):
        line = f"{prefix}: {prefix_dict[prefix]}"
        print(line)
        f.write(f"{line}\n")

print(f"\nWrote {len(prefix_dict)} unique prefixes to {output_file}")